# Scrape antifakenewscenter.com #มิจฉาชีพ news

Collect posts tagged **#มิจฉาชีพ** together with **#อาชญากรรมออนไลน์** and/or **#ข่าวปลอม**, then label them. Run the cells from top to bottom.

- **Source**: the site's public WordPress REST API (`/wp-json/wp/v2/posts?tags=<id>`), which returns the same posts as the `/tag/มิจฉาชีพ/` archive with their tag IDs.
- **Self-contained**: the scraper, the LLM labeller and its prompts are all here; only the organization list comes from `build_organizations.ipynb`.
- **Output**: `data/antifakenews_scam.csv` (`data/antifakenews_scam_test.csv` when `MAX_POSTS` is set). The last cell also saves the `call` and `convertible` posts to `data/call_and_convertable.csv`.

Output columns, in order:

| Columns | Made by |
|---|---|
| `post_id`, `date`, `hashtag`, `title`, `content` | the scrape (`date` is `DD/MM/YYYY` in the CSV) |
| `Channel` | keyword rule: `call` if the scammer phones the victim, otherwise `text` |
| `ORG` | keyword match against a list of Thai organizations (see "Organization column") |
| `scam_type`, `method_channel` | the LLM, pass 1 (see "Scam type and channel columns") |
| `call_suitable`, `summary`, `caller_role`, `goal`, `tactics`, `phrases`, `scenario_key`, `scenario_group` | the LLM, pass 2 (see "Scenario columns") |
| `source_url` | built from `post_id` |

`Channel` and `ORG` come from keyword rules, so they are a first pass, not ground truth.

In [37]:
# Imports and settings for the scrape
import html
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

API = "https://www.antifakenewscenter.com/wp-json/wp/v2"
HEADERS = {"User-Agent": "Mozilla/5.0 (research scraper; Callas project)"}
DELAY = 1.0

BASE_TAG = "มิจฉาชีพ"
TARGET_TAGS = ["อาชญากรรมออนไลน์", "ข่าวปลอม"]

MAX_POSTS = 10
OUT_PATH = Path("data/antifakenews_scam.csv" if MAX_POSTS is None else "data/antifakenews_scam_test.csv")

In [38]:
# Look up the WordPress tag IDs
def get_json(path, params, retries=3):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(f"{API}/{path}", params=params, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json(), r.headers
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(2 * attempt)


def find_tag_id(name):
    tags, _ = get_json("tags", {"search": name, "per_page": 100, "_fields": "id,name,count"})
    exact = [t for t in tags if t["name"].lstrip("#").strip() == name]
    if not exact:
        raise ValueError(f"tag not found: {name}")
    return max(exact, key=lambda t: t["count"])["id"]


tag_ids = {name: find_tag_id(name) for name in [BASE_TAG, *TARGET_TAGS]}
tag_ids

{'มิจฉาชีพ': 88, 'อาชญากรรมออนไลน์': 5713, 'ข่าวปลอม': 11}

In [39]:
# Fetch every post under the base tag
def fetch_posts(tag_id, max_pages=None):
    posts, page, total_pages = [], 1, 1
    while page <= total_pages and (max_pages is None or page <= max_pages):
        batch, headers = get_json("posts", {"tags": tag_id, "per_page": 100, "page": page,
                                            "_fields": "id,date,title,content,tags"})
        total_pages = int(headers["X-WP-TotalPages"])
        posts += batch
        print(f"page {page}/{total_pages}: {len(posts)}/{headers['X-WP-Total']} posts")
        page += 1
        if page <= total_pages:
            time.sleep(DELAY)
    return posts


raw_posts = fetch_posts(tag_ids[BASE_TAG], max_pages=None if MAX_POSTS is None else 1)

page 1/9: 100/895 posts


In [40]:
# Clean the posts into a dataframe with a Channel column
FOOTER_HEADING = "ช่องทางการติดตามและแจ้งเบาะแสข่าวปลอม"
FOOTER_START = "LINE : @antifakenewscenter"

CALL = (r"คอลเซ็นเตอร์|ทางโทรศัพท์|ผ่านโทรศัพท์|(?i:\bcall\s*center\b)|รับสาย"
        r"|โทรศัพท์(?:มา|หา|ติดต่อ|สอบ|แจ้ง|ถาม|ติดตาม)"
        r"|โทร(?:มา|หา|ไป|แจ้ง|ติดต่อ|สอบ|ถาม|ตรวจสอบ|หลอก|ขู่|เข้า)")
HOTLINE = r"(?i:call\s*center)\s*\d{3,4}|สายด่วน\s*\d+"


def clean_html(fragment):
    soup = BeautifulSoup(fragment, "html.parser")
    for tag in soup(["script", "style", "iframe", "figure", "img"]):
        tag.decompose()
    lines = (re.sub(r"[ \t]+", " ", line).strip() for line in soup.get_text("\n").replace("\xa0", " ").splitlines())
    lines = [line for line in lines if line]
    for i, line in enumerate(lines):
        if line.startswith(FOOTER_START):
            lines = lines[:i]
            if lines and lines[-1] == FOOTER_HEADING:
                lines.pop()
            break
    return "\n".join(lines)


def label_channel(text):
    return "call" if re.search(CALL, re.sub(HOTLINE, "", text)) else "text"


rows = []
for p in raw_posts:
    matched = [name for name in TARGET_TAGS if tag_ids[name] in p["tags"]]
    if matched:
        rows.append({
            "post_id": p["id"],
            "date": p["date"],
            "hashtag": ", ".join(matched),
            "title": html.unescape(BeautifulSoup(p["title"]["rendered"], "html.parser").get_text()).strip(),
            "content": clean_html(p["content"]["rendered"]),
        })

df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df["Channel"] = (df["title"] + "\n" + df["content"]).map(label_channel)
df = df.sort_values("date", ascending=False, ignore_index=True)
df

,post_id,date,hashtag,title,content,Channel
0,94831,2026-11-17,ข่าวปลอม,เพจ “สำนักงานประสานงานคุ้มครองผู้เสียหายอาชญาก...,จากการตรวจสอบ ยืนยันเป็นเพจปลอม พบมีการแอบอ้าง...,text
1,94824,2026-11-17,อาชญากรรมออนไลน์,เช็กให้ชัวร์ แอปเงินกู้ ถูกกฎหมายหรือเถื่อน?,กู้เงินออนไลน์สะดวก รวดเร็ว แต่รู้หรือไม่ว่า แ...,text
2,94815,2026-11-16,ข่าวปลอม,นายกฯ ส่งลิงก์ แจกอินเทอร์เน็ต 60GB เนื่องในวั...,ขอให้ประชาชนโปรดระวัง!! หลังพบการแอบอ้างเป็นนา...,text
3,94790,2026-11-16,ข่าวปลอม,ตำรวจเพิ่มช่องทางติดต่อ เปิดเพจเฟซบุ๊ก ปภบ.ป้อ...,เพจเฟซบุ๊กดังกล่าวเป็นเพจปลอม ไม่ใช่ช่องทางประ...,text
4,94780,2026-11-16,อาชญากรรมออนไลน์,บริษัทปลอมหลอกให้เซ็นสัญญาจ้างออนไลน์,ได้งานออนไลน์ อย่าเพิ่งรีบเซ็น! ระวังมิจฉาชีพอ...,text
...,...,...,...,...,...,...
72,90359,2026-07-29,ข่าวปลอม,การไฟฟ้าส่วนภูมิภาค เปิดบัญชี TikTok ใช้ชื่อ g...,จากการตรวจสอบ บัญชี TikTok ที่ใช้ชื่อว่า gupup...,text
73,90229,2026-07-26,อาชญากรรมออนไลน์,หลอกขายโหลจุ่มเครื่องประดับ,กระแสโหลจุ่มเครื่องประดับยังคงได้รับความนิยม ท...,text
74,90146,2026-07-24,อาชญากรรมออนไลน์,มิจแนะนำจองที่พักปลอมกับเพจลวง,ประกาศ “ห้องหลุดจอง” หรือ “โปรราคาพิเศษเฉพาะวั...,text
75,89980,2026-07-20,อาชญากรรมออนไลน์,มิจโพสต์รับแลกเงินสกุลต่างประเทศ,วันหยุดยาวนี้ ใครที่กำลังวางแผนไปเที่ยวต่างประ...,text


In [41]:
# Drop empty posts and check the scrape
empty = (df["title"] == "") | (df["content"] == "")
if empty.any():
    print(f"dropping {empty.sum()} posts with no title or content: {df.loc[empty, 'post_id'].tolist()}")
df = df[~empty].reset_index(drop=True)

if MAX_POSTS is not None:
    df = df.head(MAX_POSTS)
    print(f"test run: kept the {len(df)} newest posts")

assert df["post_id"].is_unique
assert df["date"].notna().all()
assert df["hashtag"].str.contains("|".join(TARGET_TAGS)).all()
assert not df["content"].str.contains(FOOTER_START, regex=False).any()
print(f"{len(df)} posts passed the checks")

test run: kept the 10 newest posts
10 posts passed the checks


## Organization column

`ORG` names the Thai organization a post mentions as the one the scammer pretended to be. It is matched against `data/thai_organizations.csv` (agencies, ministries and banks), which `build_organizations.ipynb` builds from DGA open data, Thai Wikipedia and Wikidata. Run that notebook first, and again only for a fresh list.

- **Aliases**: full name, abbreviation, `ธ.`/`แบงก์` + bank name, English name. Spaces are removed before matching because Thai spacing is irregular.
- **Title first**: every organization named in the title, in order.
- **Otherwise the content**: the most mentioned organization, ignoring the police, since almost every post tells the reader to report to them.
- **Limits**: private companies (HomePro, Lotus's, Singha ...) and local bodies are not in the list, so those posts get an empty `ORG`. A match means the post *mentions* the organization, not that the scammer impersonated it.

In [42]:
# Load the organization list
ORG_PATH = Path("data/thai_organizations.csv")
orgs = pd.read_csv(ORG_PATH, dtype=str, encoding="utf-8-sig", keep_default_na=False)

has_thai = lambda s: bool(re.search(r"[฀-๿]", s))

print(f"{len(orgs)} organizations from {ORG_PATH}; {(orgs['abbr_en'] != '').sum()} with an English abbreviation")

451 organizations from data\thai_organizations.csv; 80 with an English abbreviation


In [43]:
# Add the ORG column by matching organization names
def squash(text):
    # Drop Thai spacing (it is irregular) but keep the gap between two Latin words.
    text = text.replace("​", "")

    def gap(m):
        before = text[m.start() - 1] if m.start() else ""
        after = text[m.end()] if m.end() < len(text) else ""
        return " " if re.match(r"[A-Za-z0-9]", before) and re.match(r"[A-Za-z0-9]", after) else ""

    return re.sub(r"\s+", gap, text)


MANUAL_ALIASES = {
    "สำนักงานตำรวจแห่งชาติ": ["ตำรวจ", "ตร."],
    "กรมสอบสวนคดีพิเศษ": ["ดีเอสไอ"],
    "ธนาคารแห่งประเทศไทย": ["แบงก์ชาติ"],
    "ธนาคารทหารไทยธนชาต": ["ทีทีบี"],
}
AMBIGUOUS_BANK_BARE = {"กรุงเทพ", "ไทย", "เครดิต"}
GENERIC_COMPANY_BASE = {"ขนส่ง", "การบิน"}
BLOCKED_ALIASES = {"SME"}
IGNORE_IN_CONTENT = {"สำนักงานตำรวจแห่งชาติ"}


def org_aliases(row):
    name = row.name_th
    base = re.sub(r"\s*\((?:มหาชน|องค์การมหาชน)\)|\s*จำกัด|^บริษัท\s*", "", name).strip()
    out = {name, *MANUAL_ALIASES.get(name, []), *filter(None, row.abbr_en.split("|"))}
    if len(squash(base)) >= 6 and base not in GENERIC_COMPANY_BASE:
        out.add(base)
    if base.startswith("ธนาคาร"):
        rest = base[len("ธนาคาร"):].strip()
        out |= {"ธ." + rest, "แบงก์" + rest}
        if len(rest) >= 4 and rest not in AMBIGUOUS_BANK_BARE:
            out.add(rest)
    for a in filter(None, row.abbr_th.split("|")):
        out |= {a, re.sub(r"^สำนักงาน\s*", "", a)}
    return {a for a in out if a not in BLOCKED_ALIASES and len(squash(a)) >= (4 if has_thai(a) else 3)}


ALIAS_TO_ORG = {}
for row in orgs.itertuples():
    for alias in org_aliases(row):
        ALIAS_TO_ORG.setdefault(squash(alias), row.name_th)

ALIAS_RE = re.compile("|".join(
    re.escape(a) if has_thai(a) else rf"(?<![A-Za-z]){re.escape(a)}(?![A-Za-z])"
    for a in sorted(ALIAS_TO_ORG, key=len, reverse=True)
))


def org_mentions(text):
    # Organizations named in the text, in order of appearance.
    return [ALIAS_TO_ORG[m.group()] for m in ALIAS_RE.finditer(squash(text))]


def label_organization(title, content):
    in_title = dict.fromkeys(org_mentions(title))
    if in_title:
        return ", ".join(in_title)
    counts = Counter(org for org in org_mentions(content) if org not in IGNORE_IN_CONTENT)
    return counts.most_common(1)[0][0] if counts else ""


df["ORG"] = [label_organization(t, c) for t, c in zip(df["title"], df["content"])]
labelled = df.loc[df["ORG"] != "", "ORG"]
print(f"{len(labelled)}/{len(df)} posts have an organization")
labelled.str.split(", ").explode().value_counts().head(20)

3/10 posts have an organization


ORG
สำนักงานตำรวจแห่งชาติ    2
ธนาคารแห่งประเทศไทย      1
Name: count, dtype: int64

## LLM labelling engine

Two passes label the posts on the project's vLLM server: first the scam type and channel, then the scenario columns. The cell below holds the code both share.

- **Connection**: `VLLM_BASE_URL`, `VLLM_API_KEY` and `VLLM_MODEL` from the repo-root `.env`. Thinking is turned off so the tokens go to the answer.
- **Batches**: posts are sent several at a time, each tagged `[post_id]`; the model answers `{"posts": [{"post_id": 78875, ...}]}` and the ids map the answers back to rows. A bad or cut-off reply splits the batch in half, posts missing from a reply are asked once more, and posts that still fail get empty fields and `llm_error = True`.
- **Checkpoints**: answers are appended to `data/llm_<pass>_checkpoint.jsonl`, so a stopped run resumes where it left off. The file is deleted once every post is labelled. Delete it by hand after editing a prompt, or the old answers are reused.
- **Prompts**: each pass keeps its own as Python strings. `{name}` placeholders are filled with `str.replace`, so braces inside a post cannot break the prompt.

In [44]:
# Shared engine for the LLM labelling passes
import json
import os
import sys
from dataclasses import dataclass
from typing import Callable

import yaml
from dotenv import load_dotenv
from openai import OpenAI

REPO = Path.cwd().parent
load_dotenv(REPO / ".env")

sys.path.insert(0, str(REPO / "modules"))
from data_pipeline.config import SLOT_NAMES 

MAX_CONTENT_CHARS = 3000

 # Replace {key} placeholders with str.replace, so braces in a post cannot break the prompt.
def fill(template, **values):
   
    for key, value in values.items():
        template = template.replace("{" + key + "}", value)
    return template


@dataclass
class Task:

    name: str
    system_prompt: str
    user_prompt: str
    columns: list
    fields: list
    clean: Callable
    tokens_per_post: int
    batch_size: int

    @property
    def checkpoint(self):
        return Path("data") / f"llm_{self.name}_checkpoint.jsonl"

#Client for the vLLM server in .env.
def get_client():
    
    return OpenAI(base_url=os.environ["VLLM_BASE_URL"], api_key=os.environ.get("VLLM_API_KEY") or "EMPTY",
                  timeout=600, max_retries=2)

#One chat call. Asks for JSON mode, and drops it if the server does not support it.
def ask(client, task, user_text, n_posts):

    request = dict(
        model=os.environ["VLLM_MODEL"],
        messages=[{"role": "system", "content": task.system_prompt}, {"role": "user", "content": user_text}],
        temperature=0,
        max_tokens=task.tokens_per_post * n_posts,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    try:
        reply = client.chat.completions.create(**request, response_format={"type": "json_object"})
    except Exception as error:
        if "response_format" not in str(error).lower():
            raise
        reply = client.chat.completions.create(**request)
    choice = reply.choices[0]
    if not choice.message.content:
        raise ValueError(f"empty reply (finish_reason={choice.finish_reason})")
    if choice.finish_reason == "length":
        raise ValueError("reply cut off at max_tokens")
    return choice.message.content

# Turn the model's reply into {post_id: answer} for the posts it answered.
def parse_reply(task, raw):
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"no JSON in reply: {raw[:300]!r}")
    items = json.loads(match.group(0)).get("posts", [])
    return {str(item.get("post_id", "")).strip(): task.clean(item) for item in items}


# A post's values as prompt text, with the content cut to MAX_CONTENT_CHARS.
def prompt_values(post):
    values = {key: str(value) for key, value in post.items()}
    values["content"] = values["content"][:MAX_CONTENT_CHARS]
    return values

# Label a list of posts (dicts of the task's columns, lower-cased) in one request. Returns {post_id: answer}.
def label_batch(client, task, posts, retry=True):
    message = "\n\n".join(fill(task.user_prompt, **prompt_values(post)) for post in posts)
    try:
        answers = parse_reply(task, ask(client, task, message, len(posts)))
    except ValueError as error:
        if len(posts) > 1:
            print(f"bad reply for {len(posts)} posts ({error}); splitting the batch")
            half = len(posts) // 2
            return {**label_batch(client, task, posts[:half], retry),
                    **label_batch(client, task, posts[half:], retry)}
        print(f"bad reply: {error}")
        answers = {}
    except Exception as error:
        print(f"request failed: {error}")
        answers = {}

    results = {post["post_id"]: {**answers[str(post["post_id"])], "llm_error": False}
               for post in posts if str(post["post_id"]) in answers}
    missing = [post for post in posts if post["post_id"] not in results]
    if missing and retry:
        results.update(label_batch(client, task, missing, retry=False))
    else:
        for post in missing:
            print(f"failed: [{post['post_id']}] {str(post['title'])[:60]}")
            results[post["post_id"]] = {**dict.fromkeys(task.fields, ""), "llm_error": True}
    return results

# Answers saved by an earlier run that stopped early, as {post_id (str): answer}.
def load_checkpoint(task):
    if not task.checkpoint.exists():
        return {}
    saved = {}
    for line in task.checkpoint.read_text(encoding="utf-8").splitlines():
        try:
            item = json.loads(line)
            saved[item["post_id"]] = item["answer"]
        except (ValueError, KeyError):
            pass
    return saved

#Label every row of df with one task. Returns the task's fields + llm_error, same index."""
def label_all(df, task):
    client = get_client()
    labels = load_checkpoint(task)
    if labels:
        print(f"resuming: {len(labels)} answers found in {task.checkpoint.name}")
    rows = df[task.columns].rename(columns=str.lower).to_dict("records")
    posts = [post for post in rows if str(post["post_id"]) not in labels]

    task.checkpoint.parent.mkdir(parents=True, exist_ok=True)
    with task.checkpoint.open("a", encoding="utf-8") as checkpoint:
        for start in range(0, len(posts), task.batch_size):
            for post_id, answer in label_batch(client, task, posts[start:start + task.batch_size]).items():
                labels[str(post_id)] = answer
                if not answer["llm_error"]:
                    checkpoint.write(json.dumps({"post_id": str(post_id), "answer": answer}, ensure_ascii=False) + "\n")
            checkpoint.flush()
            print(f"labelled {min(start + task.batch_size, len(posts))}/{len(posts)}")

    result = pd.DataFrame([labels[str(post_id)] for post_id in df["post_id"]], index=df.index)
    if not result["llm_error"].any():
        task.checkpoint.unlink()
    return result


print(f"vLLM server: {os.environ.get('VLLM_BASE_URL')} | model: {os.environ.get('VLLM_MODEL')}")

vLLM server: https://llm.vitawat.dev/llm/v1 | model: gemma-4-31b-it-turbo


## Scam type and channel columns (labelled by the LLM, pass 1)

Pass 1 labels every post, 50 per request:

- `scam_type`: one of `fake_authority`, `romance`, `investment`, `phishing_otp`, `lottery`, or empty when the post describes no scam (an overview of several types, a fact-check of a rumour).
- `method_channel`: where the scam happens (see "Channel column" below).

The cells hold the prompts (the scam type definitions are sent whole, so editing them changes the labels), the answer check and setup, and the run.

In [45]:
# Prompts for pass 1: scam type and channel
SCAM_TYPE_DEFINITIONS = """# Scam type definitions

How the `scam_type` column in `data/antifakenews_scam.csv` was labelled. The five types are the
Callas scam types, with wider definitions so that every scam on antifakenewscenter.com fits one.

This text is sent to the model as its instructions, so a change here changes the labels.

## Types

| Type | Definition |
|---|---|
| `fake_authority` | Pretends to be a trusted organization or person in an official role (government agency, police, bank, utility, post or courier, company or shop), often through a fake page or account in its name. It threatens the victim, or offers a service such as a loan, a driving licence, money recovery or a job, or a government handout, welfare, stimulus or refund payment in an official's name, then demands a fee, fine, transfer, or personal data such as an ID card or bank account number. |
| `romance` | Builds trust as a partner, friend, relative or acquaintance, or takes over or copies the account of someone the victim knows or a celebrity, then asks to borrow money or for a transfer. |
| `investment` | Promises high or guaranteed returns from stocks, crypto, trading, funds or paid "task" jobs. Shows fake profits, then blocks withdrawals until more "fees" are paid. |
| `phishing_otp` | Pretends to be a bank, courier, company or official, and the main ask is an OTP, password or PIN, or to click a link (SMS, QR code, fake website) or install an app from one, including remote-control apps. A request only for personal data, an ID card or a bank account number, with no code or link, is `fake_authority`. |
| `lottery` | Claims the victim won a prize, gift, lucky draw or lottery tip, from a company, shop, draw or celebrity, then asks for a fee or bank details before they can receive it. A government handout, welfare or stimulus payment offered in an official's name is `fake_authority`, not `lottery`. |

## Tie-break

When a post fits several types, the first match in this order wins:

1. `romance`: the scam relies on a personal relationship or trust in someone the victim knows.
2. `investment`: the lure is profit or returns.
3. `lottery`: the lure is a prize, a draw or something free from a company or draw (not a government payment).
4. `phishing_otp`: the main ask is an OTP, password or PIN, a link or an app.
5. `fake_authority`: everything else that uses impersonation to get paid.

## Channel field

`method_channel` says where the scam happens, from a fixed list, comma separated
(e.g. post 78875: เพจเฟซบุ๊ก).

Channel values: เพจเฟซบุ๊ก, กลุ่มเฟซบุ๊ก, Messenger, TikTok, LINE, SMS, โทรศัพท์, เว็บไซต์, อีเมล, เอกสาร,
แอปพลิเคชัน, โซเชียลมีเดีย (social media, platform not named), พอดแคสต์, QR Code, Google Form."""

LABEL_SYSTEM = """You label Thai scam-warning posts from antifakenewscenter.com. You get many posts at once, each
starting with its post_id in square brackets, e.g. [78875]. Label every post on its own.

{definitions}

Rules:
- Pick exactly one scam_type with the definitions and tie-break above. Use "" (empty) when the post
  describes no scam, e.g. an overview of several scam types or a fact-check of a rumour.
- method_channel: only values from the channel list, several joined with ", ".
- Never invent a detail the post does not state; leave that field "".
- When scam_type is "", method_channel is "".

Reply with JSON only, no other text: one object with a "posts" list that has one item for every
post, in order, where "post_id" is the id from the post's square brackets:
{"posts": [{"post_id": 78875, "scam_type": "", "method_channel": ""}]}"""

LABEL_USER = """[{post_id}]
Title: {title}
Content: {content}"""

In [46]:
# Answer cleaning and task setup for pass 1
from nlp_core.types import SCAM_TYPES

CHANNELS = {
    "เพจเฟซบุ๊ก", "กลุ่มเฟซบุ๊ก", "Messenger", "TikTok", "LINE", "SMS", "โทรศัพท์", "เว็บไซต์", "อีเมล",
    "เอกสาร", "แอปพลิเคชัน", "โซเชียลมีเดีย", "พอดแคสต์", "QR Code", "Google Form",
}
FIELDS = ["scam_type", "method_channel"]


# Keep only allowed values for one post's scam type answer.
def clean(item):
    scam_type = str(item.get("scam_type") or "").strip()
    if scam_type not in SCAM_TYPES:
        return {"scam_type": "", "method_channel": ""}
    channels = str(item.get("method_channel") or "").split(",")
    return {"scam_type": scam_type,
            "method_channel": ", ".join(c.strip() for c in channels if c.strip() in CHANNELS)}


LABEL_TASK = Task(
    name="labels",
    system_prompt=fill(LABEL_SYSTEM, definitions=SCAM_TYPE_DEFINITIONS),
    user_prompt=LABEL_USER,
    columns=["post_id", "title", "content"],
    fields=FIELDS,
    clean=clean,
    tokens_per_post=100,
    batch_size=50,
)

In [47]:
# Run pass 1 and add scam_type and method_channel
labels = label_all(df, LABEL_TASK)
df[FIELDS] = labels[FIELDS]

print(f"{labels['llm_error'].sum()} posts failed to label")
df["scam_type"].replace("", "(none)").value_counts()

labelled 10/10
0 posts failed to label


scam_type
fake_authority    7
(none)            2
phishing_otp      1
Name: count, dtype: int64

## Channel column

`method_channel` says where each scam happens, from a fixed list (เพจเฟซบุ๊ก, กลุ่มเฟซบุ๊ก, TikTok, LINE, SMS, โทรศัพท์, เว็บไซต์, อีเมล, เอกสาร ...), comma separated.

- It comes from the same LLM call as `scam_type`.
- Channels outside the list are dropped, and posts with no `scam_type` have it empty.
- A post that lists โทรศัพท์ is a phone scam: `call_suitable` = `call` (see "Scenario columns").

In [48]:
# Preview the method_channel column
df[["title", "method_channel"]].head()

,title,method_channel
0,เพจ “สำนักงานประสานงานคุ้มครองผู้เสียหายอาชญาก...,เพจเฟซบุ๊ก
1,เช็กให้ชัวร์ แอปเงินกู้ ถูกกฎหมายหรือเถื่อน?,
2,นายกฯ ส่งลิงก์ แจกอินเทอร์เน็ต 60GB เนื่องในวั...,โซเชียลมีเดีย
3,ตำรวจเพิ่มช่องทางติดต่อ เปิดเพจเฟซบุ๊ก ปภบ.ป้อ...,เพจเฟซบุ๊ก
4,บริษัทปลอมหลอกให้เซ็นสัญญาจ้างออนไลน์,"โซเชียลมีเดีย, เอกสาร"


## Scenario columns (labelled by the LLM, pass 2)

Pass 2 runs only on posts with a `scam_type`, 20 per request, and describes each scam the way the call simulator (`modules/data_pipeline`) needs it:

| Column | What it holds |
|---|---|
| `call_suitable` | `call` (the scammer phones; always `call` when `method_channel` lists โทรศัพท์), `convertible` (could be told in a call) or `no` |
| `summary` | one English line naming the pattern |
| `caller_role` | who the scammer claims to be, in English |
| `goal` | what the scammer wants by the end, in English, with slots such as `{MONEY}` |
| `tactics` | the scam's steps in order, in English, joined with ` \| ` |
| `phrases` | Thai lines the post quotes from the scammer, with slots, joined with ` \| ` |
| `scenario_key` | the matching scenario in the simulator's `scenarios.yaml`, or `new` |
| `scenario_group` | a short name shared by posts with the same new situation |
| `source_url` | the post's link, built from `post_id` (no LLM) |

- **Prompts**: `{scenarios}` is filled with the simulator's existing scam scenarios, `{slots}` with its slot names.
- **Answer check**: real numbers, links and LINE IDs the model left in are turned into the simulator's slots.
- Posts with no `scam_type` have these columns empty (except `source_url`).

In [49]:
# Prompts for pass 2: scenario columns
SCENARIO_SYSTEM = """You read Thai scam-warning posts from antifakenewscenter.com and describe each scam so that a
phone-call simulator can imitate it to train a scam-detection system. You get many posts at once,
each starting with its post_id in square brackets, e.g. [78875], then its scam_type, organization,
title and content. Describe every post on its own.

Fields for each post:
- call_suitable: "call" if the post says the scammer phones or voice-calls the victim (a LINE
  or video call counts). "convertible"
  if it is not a phone call but the same story could be told in one (an official, a bill, a
  parcel, a prize, a relative asking for help). "no" if the scam depends on something seen or
  read (a fake page or ad, a QR code, a document, a shop listing, a fake website alone).
- summary: one short English line naming the pattern, e.g. "Caller poses as the post office and
  asks for a redelivery fee".
- caller_role: who the scammer claims to be, in English, e.g. "officer of the electricity authority".
- goal: one English sentence: what the scammer wants the victim to do by the end, with slots for
  amounts, accounts, links, codes and phone numbers, e.g. "get the receiver to pay {MONEY} to
  {BANK_ACCOUNT}".
- tactics: a list of 2 to 5 short English steps, in the order the scam goes.
- phrases: a list of Thai lines the post quotes as said or written by the scammer (in quotes, or
  after อ้างว่า / แจ้งว่า / บอกว่า). Copy the post's words exactly, only replacing numbers, amounts,
  links, account numbers and names with slots. Use [] when the post quotes nothing.
- scenario_key: the key of the existing scenario below with the same scam_type and the same
  situation (the same kind of caller asking for the same kind of thing), or "new".
- scenario_group: when scenario_key is "new", a short snake_case English name for the situation,
  e.g. "post_redelivery_fee"; give posts with the same situation the same name. Otherwise repeat
  the scenario_key.

Existing scenarios (scam_type / key: role; wants: goal):
{scenarios}

Rules:
- Write in English, except phrases, which stay in Thai.
- Never write a real phone number, account number, amount, link, LINE ID or person's name: use a
  slot instead. The only slots are: {slots}
- Never invent a detail the post does not state; leave that field "" or [].

Reply with JSON only, no other text: one object with a "posts" list that has one item for every
post, in order, where "post_id" is the id from the post's square brackets:
{"posts": [{"post_id": 78875, "call_suitable": "", "summary": "", "caller_role": "", "goal": "", "tactics": [], "phrases": [], "scenario_key": "new", "scenario_group": ""}]}"""

SCENARIO_USER = """[{post_id}]
scam_type: {scam_type}
Organization: {org}
Title: {title}
Content: {content}"""

In [50]:
# Answer cleaning and task setup for pass 2
SCENARIO_FIELDS = [
    "call_suitable", "summary", "caller_role", "goal", "tactics", "phrases", "scenario_key", "scenario_group",
]
SUITABLE = {"call", "convertible", "no"}
LIST_SEP = " | "
SCENARIOS_YAML = REPO / "modules" / "data_pipeline" / "config" / "scenarios.yaml"
POST_URL = "https://www.antifakenewscenter.com/?p={}"

TO_SLOTS = [
    (re.compile(r"https?://\S+|www\.\S+|\b[\w-]+(?:\.[\w-]+)*\.(?:com|net|org|co|ly|me|th|info|xyz)\b\S*"), "{URL}"),
    (re.compile(r"[\w-]+\.\{URL\}"), "{URL}"),
    (re.compile(r"\d[\d,.]*\s*(?:บาท|baht)", re.IGNORECASE), "{MONEY}"),
    (re.compile(r"\b0\d{1,2}[- ]?\d{3}[- ]?\d{3,4}\b"), "{PHONE}"),
    (re.compile(r"@[^\s,()\"']{3,}"), "{URL}"),
]
SLOT = re.compile(r"\{([A-Za-z_]+)\}")


# Replace real numbers, links and LINE IDs with slots, and unwrap slots the simulator lacks.
def to_slots(text):
    for pattern, slot in TO_SLOTS:
        text = pattern.sub(slot, text)
    return SLOT.sub(lambda m: m.group(0) if m.group(1) in SLOT_NAMES else m.group(1), text)


# The model may answer a list or one string; always give a list of non-empty strings.
def as_list(value):
    items = value if isinstance(value, list) else str(value or "").split("|")
    return [str(item).strip() for item in items if str(item).strip()]


SCENARIOS = yaml.safe_load(SCENARIOS_YAML.read_text(encoding="utf-8"))["scams"]
SCENARIO_KEYS = {s["key"] for scenarios in SCENARIOS.values() for s in scenarios}


# Keep only allowed values for one post's scenario answer, with slots in place of numbers.
def clean_scenario(item):
    def text(field):
        return str(item.get(field) or "").strip()

    suitable = text("call_suitable").lower()
    key = text("scenario_key")
    key = key if key in SCENARIO_KEYS else "new"
    group = re.sub(r"[^a-z0-9_]+", "_", text("scenario_group").lower()).strip("_")
    return {
        "call_suitable": suitable if suitable in SUITABLE else "",
        "summary": to_slots(text("summary")),
        "caller_role": to_slots(text("caller_role")),
        "goal": to_slots(text("goal")),
        "tactics": LIST_SEP.join(to_slots(t) for t in as_list(item.get("tactics"))),
        "phrases": LIST_SEP.join(to_slots(p) for p in as_list(item.get("phrases"))),
        "scenario_key": key,
        "scenario_group": group if key == "new" else key,
    }


# The existing scenarios as prompt text, one line each: type, key, role and goal.
def scenario_list_text():
    return "\n".join(
        f"- {scam_type} / {s['key']}: {s['role']}; wants: {' '.join(s['goal'].split())}"
        for scam_type, scenarios in SCENARIOS.items()
        for s in scenarios
    )


SCENARIO_TASK = Task(
    name="scenarios",
    system_prompt=fill(SCENARIO_SYSTEM, scenarios=scenario_list_text(),
                       slots=" ".join("{" + s + "}" for s in SLOT_NAMES)),
    user_prompt=SCENARIO_USER,
    columns=["post_id", "scam_type", "ORG", "title", "content"],
    fields=SCENARIO_FIELDS,
    clean=clean_scenario,
    tokens_per_post=700,
    batch_size=20,
)


# Run the scenario pass on the posts that have a scam_type and add its columns to df.
# Posts with no scam_type get empty scenario fields. Also adds source_url (every post).
def add_scenario_columns(df):
    scams = df[df["scam_type"] != ""]
    labels = label_all(scams, SCENARIO_TASK)
    df[SCENARIO_FIELDS] = ""
    df.loc[scams.index, SCENARIO_FIELDS] = labels[SCENARIO_FIELDS]
    phone = df["method_channel"].str.contains("โทรศัพท์")
    df.loc[phone, "call_suitable"] = "call"
    df["source_url"] = df["post_id"].map(POST_URL.format)
    print(f"{labels['llm_error'].sum()} of {len(scams)} scam posts failed to label")
    return df

In [51]:
# Run pass 2 and add the scenario columns
df = add_scenario_columns(df)
df["call_suitable"].replace("", "(no scam)").value_counts()

labelled 8/8
0 of 8 scam posts failed to label


call_suitable
convertible    6
(no scam)      2
no             2
Name: count, dtype: int64

In [52]:
# Summarise how many posts were kept
print(f"{len(raw_posts)} posts under #{BASE_TAG} -> {len(df)} kept")
display(df["hashtag"].value_counts())
df["Channel"].value_counts()

100 posts under #มิจฉาชีพ -> 10 kept


hashtag
ข่าวปลอม            6
อาชญากรรมออนไลน์    4
Name: count, dtype: int64

Channel
text    10
Name: count, dtype: int64

In [53]:
# Save the dataset to CSV and read it back
# Save the dataset as UTF-8 with BOM (so Excel shows the Thai text), dates as DD/MM/YYYY.
def save_dataset(df, path):
    if pd.api.types.is_datetime64_any_dtype(df["date"]):
        df = df.assign(date=df["date"].dt.strftime("%d/%m/%Y"))
    df.to_csv(path, index=False, encoding="utf-8-sig")


OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
save_dataset(df, OUT_PATH)
print(f"saved {len(df)} posts to {OUT_PATH}")

pd.read_csv(OUT_PATH, parse_dates=["date"], date_format="%d/%m/%Y", keep_default_na=False).head()

saved 10 posts to data\antifakenews_scam_test.csv


,post_id,date,hashtag,title,content,Channel,ORG,scam_type,method_channel,call_suitable,summary,caller_role,goal,tactics,phrases,scenario_key,scenario_group,source_url
0,94831,2026-11-17,ข่าวปลอม,เพจ “สำนักงานประสานงานคุ้มครองผู้เสียหายอาชญาก...,จากการตรวจสอบ ยืนยันเป็นเพจปลอม พบมีการแอบอ้าง...,text,,fake_authority,เพจเฟซบุ๊ก,convertible,Scammer creates a fake cybercrime victim prote...,cyber police officer,get the receiver to provide personal informati...,Create a fake Facebook page with a name simila...,,new,fake_police_support_page,https://www.antifakenewscenter.com/?p=94831
1,94824,2026-11-17,อาชญากรรมออนไลน์,เช็กให้ชัวร์ แอปเงินกู้ ถูกกฎหมายหรือเถื่อน?,กู้เงินออนไลน์สะดวก รวดเร็ว แต่รู้หรือไม่ว่า แ...,text,ธนาคารแห่งประเทศไทย,,,,,,,,,,,https://www.antifakenewscenter.com/?p=94824
2,94815,2026-11-16,ข่าวปลอม,นายกฯ ส่งลิงก์ แจกอินเทอร์เน็ต 60GB เนื่องในวั...,ขอให้ประชาชนโปรดระวัง!! หลังพบการแอบอ้างเป็นนา...,text,,phishing_otp,โซเชียลมีเดีย,no,Scammer impersonates the Prime Minister to off...,Prime Minister,get the receiver to click {URL} and provide pe...,Send a message claiming to be the Prime Minist...,แจกอินเทอร์เน็ต 60GB เนื่องในวันชาติ,new,fake_pm_internet_giveaway,https://www.antifakenewscenter.com/?p=94815
3,94790,2026-11-16,ข่าวปลอม,ตำรวจเพิ่มช่องทางติดต่อ เปิดเพจเฟซบุ๊ก ปภบ.ป้อ...,เพจเฟซบุ๊กดังกล่าวเป็นเพจปลอม ไม่ใช่ช่องทางประ...,text,สำนักงานตำรวจแห่งชาติ,fake_authority,เพจเฟซบุ๊ก,convertible,Scammer creates a fake police cyber-warning pa...,police officer,get the receiver to provide personal informati...,Create a fake Facebook page impersonating a po...,,new,fake_police_support_page,https://www.antifakenewscenter.com/?p=94790
4,94780,2026-11-16,อาชญากรรมออนไลน์,บริษัทปลอมหลอกให้เซ็นสัญญาจ้างออนไลน์,ได้งานออนไลน์ อย่าเพิ่งรีบเซ็น! ระวังมิจฉาชีพอ...,text,,fake_authority,"โซเชียลมีเดีย, เอกสาร",convertible,Scammer poses as HR to trick job seekers into ...,HR officer,get the receiver to provide {ID_CARD} and sign...,Contact job applicants via online channels | S...,,new,fake_job_contract_identity_theft,https://www.antifakenewscenter.com/?p=94780


In [54]:
# Keep only the posts a call can be built from, and save them
call_posts = df[df["call_suitable"].isin(["call", "convertible"])]
CALL_PATH = OUT_PATH.with_name("call_and_convertable.csv")
save_dataset(call_posts, CALL_PATH)
print(f"saved {len(call_posts)} of {len(df)} posts to {CALL_PATH}: {call_posts['call_suitable'].value_counts().to_dict()}")
call_posts

saved 6 of 10 posts to data\call_and_convertable.csv: {'convertible': 6}


,post_id,date,hashtag,title,content,Channel,ORG,scam_type,method_channel,call_suitable,summary,caller_role,goal,tactics,phrases,scenario_key,scenario_group,source_url
0,94831,2026-11-17,ข่าวปลอม,เพจ “สำนักงานประสานงานคุ้มครองผู้เสียหายอาชญาก...,จากการตรวจสอบ ยืนยันเป็นเพจปลอม พบมีการแอบอ้าง...,text,,fake_authority,เพจเฟซบุ๊ก,convertible,Scammer creates a fake cybercrime victim prote...,cyber police officer,get the receiver to provide personal informati...,Create a fake Facebook page with a name simila...,,new,fake_police_support_page,https://www.antifakenewscenter.com/?p=94831
3,94790,2026-11-16,ข่าวปลอม,ตำรวจเพิ่มช่องทางติดต่อ เปิดเพจเฟซบุ๊ก ปภบ.ป้อ...,เพจเฟซบุ๊กดังกล่าวเป็นเพจปลอม ไม่ใช่ช่องทางประ...,text,สำนักงานตำรวจแห่งชาติ,fake_authority,เพจเฟซบุ๊ก,convertible,Scammer creates a fake police cyber-warning pa...,police officer,get the receiver to provide personal informati...,Create a fake Facebook page impersonating a po...,,new,fake_police_support_page,https://www.antifakenewscenter.com/?p=94790
4,94780,2026-11-16,อาชญากรรมออนไลน์,บริษัทปลอมหลอกให้เซ็นสัญญาจ้างออนไลน์,ได้งานออนไลน์ อย่าเพิ่งรีบเซ็น! ระวังมิจฉาชีพอ...,text,,fake_authority,"โซเชียลมีเดีย, เอกสาร",convertible,Scammer poses as HR to trick job seekers into ...,HR officer,get the receiver to provide {ID_CARD} and sign...,Contact job applicants via online channels | S...,,new,fake_job_contract_identity_theft,https://www.antifakenewscenter.com/?p=94780
5,94748,2026-11-15,ข่าวปลอม,เพจเฟซบุ๊ก นปขซ.-หน่วยประสานงานข้อมูลความเสียห...,เพจเฟซบุ๊กดังกล่าวเป็นเพจปลอม ไม่ใช่ช่องทางประ...,text,,fake_authority,เพจเฟซบุ๊ก,convertible,Scammer creates a fake cyber-damage coordinati...,police officer,get the receiver to provide personal informati...,Create a fake Facebook page impersonating a po...,,new,fake_police_support_page,https://www.antifakenewscenter.com/?p=94748
7,94717,2026-11-14,ข่าวปลอม,เพจ ศูนย์ประสานงานเพื่อส่งเสริมความปลอดภัยและช...,เพจเก่าปิด เพจใหม่มา สำนักงานตำรวจแห่งชาติแจ้ง...,text,สำนักงานตำรวจแห่งชาติ,fake_authority,เพจเฟซบุ๊ก,convertible,Scammer creates a fake police coordination pag...,police officer,get the receiver to provide personal information,Create a fake Facebook page impersonating the ...,,new,fake_police_support_page,https://www.antifakenewscenter.com/?p=94717
9,94638,2026-11-12,ข่าวปลอม,ผบ.ตร. เปิดเพจยื่นสิทธิขอรับทรัพย์คืนจากมิจฉาช...,เป็นการแอบอ้างโดยไม่ได้รับอนุญาต ผบ.ตร. ไม่ได้...,text,,fake_authority,เพจเฟซบุ๊ก,convertible,Scammer creates a fake page claiming the Polic...,police officer,get the receiver to provide personal informati...,Create a fake page claiming the National Polic...,,new,fake_police_asset_recovery,https://www.antifakenewscenter.com/?p=94638
